# Sprint 1 — Tumor Weight Sweep (1 epoch)
**Goal:** Run 1-epoch training at `tumor_weight` ∈ {2, 3, 5} with fixed metrics, compare AUPRC + tumor Dice + foreground fraction.

---
## 1. Setup

In [ ]:
import sys, os, json, warnings
from pathlib import Path
# pyarrow TxF workaround
os.environ["PYARROW_IGNORE_ZERO_COPY"] = "1"
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

sys.path.insert(0, str(Path.cwd().parent))

# Confirm the fixed tumor_only_dice is live
from src.metrics import tumor_only_dice
print(f"tumor_only_dice loaded from: {tumor_only_dice.__module__}")

---
## 2. Unit Test — tumor_only_dice()

In [ ]:
# Verify the filtering fix: tumor-free slices should be excluded, not zeroed.
# Hand-built 3-slice batch: [all-bg, all-fg, bg-pred-on-fg]
pred = torch.zeros(3, 1, 64, 64)
pred[1] = 1.0     # slice 1: predict all FG
pred[2] = 0.0     # slice 2: predict all BG (misses the tumor)

target = torch.zeros(3, 1, 64, 64)
target[1] = 1.0   # slice 1: true FG
target[2] = 1.0   # slice 2: true FG

td = tumor_only_dice(pred, target)
print(f"tumor_only_dice on [bg/bg, fg/fg, bg/fg]: {td:.4f}")

# Slice 0 (bg/bg): no tumor in target → excluded
# Slice 1 (fg/fg): Dice = 1.0
# Slice 2 (bg/fg): Dice = 0.0 (no predicted overlap)
# Expected = (1.0 + 0.0) / 2 = 0.5
expected = 0.5
assert abs(td - expected) < 0.01, f"Expected {expected}, got {td}"
print("PASSED — tumor_only_dice filtering fix confirmed")

---
## 3. Run Sweep — tumor_weight ∈ {2, 3, 5}

In [ ]:
from src.training.sweep_runner import run_single_config, aggregate_sweep_results

TUMOR_WEIGHTS = [2, 3, 5]
EPOCHS = 1
SEED = 42
OUTPUT_DIR = "../experiments/sprint1"

results = []
for w in TUMOR_WEIGHTS:
    print(f"\n{'='*60}")
    print(f"  Sweep: tumor_weight = {w}")
    print(f"{'='*60}")
    m = run_single_config(
        tumor_weight=float(w),
        epochs=EPOCHS,
        output_dir=OUTPUT_DIR,
        seed=SEED,
        batch_size=4,
    )
    results.append(m)

df = aggregate_sweep_results(results, output_dir=OUTPUT_DIR)
print("\nSweep complete.")

---
## 4. Sweep Results Table

In [ ]:
display_cols = [
    "tumor_weight",
    "val_tumor_dice",
    "val_dice",
    "val_auprc",
    "raw_val_auprc",
    "val_auroc",
    "val_fg_frac",
    "precision",
    "recall",
    "train_loss",
]
display_df = df[display_cols].round(4)
display_df

---
## 5. Plots — AUPRC & Tumor Dice vs Tumor Weight

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(df["tumor_weight"], df["val_auprc"], "bo-", markersize=8)
axes[0].plot(df["tumor_weight"], df["raw_val_auprc"], "go--", markersize=8, alpha=0.7, label="Raw logit AUPRC")
axes[0].set_xlabel("Tumor Weight")
axes[0].set_ylabel("AUPRC")
axes[0].set_title("AUPRC vs Tumor Weight")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(df["tumor_weight"], df["val_tumor_dice"], "mo-", markersize=8)
axes[1].set_xlabel("Tumor Weight")
axes[1].set_ylabel("Tumor Dice")
axes[1].set_title("Tumor Dice vs Tumor Weight")
axes[1].grid(True)

axes[2].plot(df["tumor_weight"], df["val_fg_frac"] * 100, "co-", markersize=8)
axes[2].set_xlabel("Tumor Weight")
axes[2].set_ylabel("Foreground Fraction (%)")
axes[2].set_title("Predicted FG% vs Tumor Weight")
axes[2].grid(True)

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "sweep_results.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Comparison Bar Chart — All Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 6a. Grouped bar: val_dice, val_tumor_dice, precision, recall
metric_names = ["val_dice", "val_tumor_dice", "precision", "recall"]
x = np.arange(len(metric_names))
width = 0.25
for i, (_, row) in enumerate(df.iterrows()):
    offset = (i - 1) * width
    vals = [row[m] for m in metric_names]
    bars = axes[0].bar(x + offset, vals, width, label=f"w={int(row['tumor_weight'])}", alpha=0.8, edgecolor="black")
    for bar, v in zip(bars, vals):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f"{v:.3f}", ha="center", va="bottom", fontsize=8)
axes[0].set_xticks(x); axes[0].set_xticklabels(["Dice", "T Dice", "Precision", "Recall"])
axes[0].set_ylabel("Score"); axes[0].set_title("Dice, Precision & Recall by Weight")
axes[0].legend(); axes[0].grid(True, axis="y")

# 6b. Grouped bar: AUPRC, raw AUPRC, AUROC, FG% (scaled)
metric_names2 = ["val_auprc", "raw_val_auprc", "val_auroc"]
x2 = np.arange(len(metric_names2))
for i, (_, row) in enumerate(df.iterrows()):
    offset = (i - 1) * width
    vals = [row[m] for m in metric_names2]
    bars = axes[1].bar(x2 + offset, vals, width, label=f"w={int(row['tumor_weight'])}", alpha=0.8, edgecolor="black")
    for bar, v in zip(bars, vals):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f"{v:.3f}", ha="center", va="bottom", fontsize=8)
# Add FG% as a secondary line overlay
ax2 = axes[1].twinx()
ax2.plot(df["tumor_weight"], df["val_fg_frac"] * 100, "D-", color="darkred", markersize=8, linewidth=2, label="FG%")
ax2.set_ylabel("Foreground Fraction (%)", color="darkred")
ax2.tick_params(axis="y", labelcolor="darkred")
axes[1].set_xticks(x2); axes[1].set_xticklabels(["AUPRC (val)", "AUPRC (raw)", "AUROC"])
axes[1].set_ylabel("Score"); axes[1].set_title("AUPRC, AUROC & FG% by Weight")
axes[1].legend(loc="upper left"); axes[1].grid(True, axis="y")
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "sweep_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7. Decision Cell

**Which weight to carry into the full 5-epoch run?**

Review the table and plots above:
- w=2: AUPRC=___, T Dice=___, FG%=___
- w=3: AUPRC=___, T Dice=___, FG%=___
- w=5: AUPRC=___, T Dice=___, FG%=___

**Was v2 background collapse caused by distribution collapse (all AUPRC low) or threshold miscalibration (AUPRC reasonable, precision/recall at default threshold bad)?**

Look at `raw_val_auprc` vs `val_auprc`: if `raw_val_auprc` is substantially higher than `val_auprc`, it suggests the model assigns higher probabilities to tumor slices (ranked correctly) but the default 0.5 threshold is too aggressive — i.e., threshold miscalibration, not collapse. If both are low, it's true distribution collapse.

**Chosen weight:** ___  
**Reason:** ___